# Analisis factorial completo: efecto del TDA y las exogenas

Este notebook responde, con **pruebas de hipotesis**, si el uso de TDA o de
variables exogenas mejora el forecasting de WPUSI01102B de forma
estadisticamente significativa.

**Metodologia:**
- Se evalua un barrido factorial amplio (idealmente las ~19,200 combinaciones)
  con TimeSeriesSplit, guardando las metricas POR FOLD.
- Las pruebas se hacen sobre los folds de CV; el **test 2022+ queda intacto**.
- Analisis: Wilcoxon pareado, Cliff's delta (tamano de efecto) y regresion
  factorial (efecto de cada factor controlando los demas).

> El factorial completo tarda ~20 min con varios nucleos. Para iterar rapido,
> usa el subconjunto reducido de la primera celda.

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import warnings; warnings.filterwarnings("ignore")
import logging
for _l in ["tensorflow","prophet","cmdstanpy"]:
    logging.getLogger(_l).setLevel(logging.ERROR)

import pandas as pd
import matplotlib.pyplot as plt

from berry_price_tda.pipelines.experiment import run_full_factorial
from berry_price_tda.pipelines.analysis import full_analysis, prepare
DATA = "../../data/interim/berry_features.csv"


## 1. Barrido factorial

Descomenta la version completa cuando quieras la corrida final (~20 min).

In [ ]:
# --- Version rapida (subconjunto) para iterar ---
df_long = run_full_factorial(
    data_path=DATA,
    models=["Ridge","ElasticNet","KNN","GradientBoosting","LinearRegression","Lasso"],
    windows=[24,36,48],
    transforms=["none","standard","diff"],
    exog_subsets=[[], ["ppi_fertilizers"], ["mxn_usd"], ["ppi_fertilizers","mxn_usd"]],
    tda_options=[False,True],
    strategies=["none","winsorize","exclude"],
    cv_splits=4, n_jobs=-1,
    checkpoint="../../data/processed/factorial_long.csv",
    verbose=True,
)

# --- Version COMPLETA (todas las ~19,200 combinaciones, ~20 min) ---
# df_long = run_full_factorial(
#     data_path=DATA, cv_splits=4, n_jobs=-1,
#     checkpoint="../../data/processed/factorial_long.csv", verbose=True,
# )
print("Filas (combo x fold):", len(df_long))
df_long.head()

## 2. Pruebas de hipotesis (Wilcoxon pareado)

In [ ]:
res = full_analysis(df_long, metric="mae", lower_is_better=True)
print("Pruebas pareadas:")
display(res["paired"])
print("\nResumen:")
print(res["summary"])

## 3. Distribucion del efecto (con vs sin cada factor)

In [ ]:
d = prepare(df_long)
fig, axes = plt.subplots(1, 2, figsize=(13,5))

# TDA
for val, color, lab in [(False,"#888780","sin TDA"), (True,"#534AB7","con TDA")]:
    sub = d[d["use_tda"]==val]["mae"].dropna()
    axes[0].hist(sub, bins=40, alpha=0.6, color=color, label=lab, density=True)
axes[0].set_title("Distribucion de MAE: con vs sin TDA")
axes[0].set_xlabel("MAE"); axes[0].legend(); axes[0].grid(alpha=0.3)

# Exogenas
for val, color, lab in [(False,"#888780","sin exog"), (True,"#1D9E75","con exog")]:
    sub = d[d["has_exog"]==val]["mae"].dropna()
    axes[1].hist(sub, bins=40, alpha=0.6, color=color, label=lab, density=True)
axes[1].set_title("Distribucion de MAE: con vs sin exogenas")
axes[1].set_xlabel("MAE"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Boxplot pareado por factor
fig, axes = plt.subplots(1, 2, figsize=(11,5))
d.boxplot(column="mae", by="use_tda", ax=axes[0])
axes[0].set_title("MAE por uso de TDA"); axes[0].set_xlabel("use_tda")
d.boxplot(column="mae", by="has_exog", ax=axes[1])
axes[1].set_title("MAE por uso de exogenas"); axes[1].set_xlabel("has_exog")
plt.suptitle(""); plt.tight_layout(); plt.show()

## 4. Regresion factorial (analisis multivariado)

Modela el MAE como funcion de TODOS los factores a la vez. El coeficiente de
cada factor es su efecto marginal controlando por los demas, con su p-valor.
Equivale a un ANOVA factorial.

In [ ]:
reg = res["regression"]
display(reg[["coef","std_err","p_value","significativo_0.05"]])

# Visualizar coeficientes significativos
sig = reg[reg["significativo_0.05"]].drop("Intercept", errors="ignore")
fig, ax = plt.subplots(figsize=(9, max(3, len(sig)*0.4)))
colors = ["#D85A30" if c>0 else "#1D9E75" for c in sig["coef"]]
ax.barh(sig.index, sig["coef"], color=colors)
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Efecto de cada factor sobre MAE (coef OLS, solo significativos)")
ax.set_xlabel("Efecto sobre MAE (negativo = mejora)")
plt.tight_layout(); plt.show()

## 5. Conclusiones

Las celdas anteriores dan el material para afirmar, con sustento estadistico:
- Si el TDA mejora/empeora/es neutral (p-valor + tamano de efecto)
- Si las exogenas aportan
- Que factores son los que mas influyen en el desempeno

Recuerda: todo esto es sobre los folds de CV. El mejor modelo final y su
desempeno honesto en test 2022+ vienen de `run_optuna.py`.